In [ ]:
# ─── Celda 1: Train / Test split ─────────────────────────────────────────────
from sklearn.model_selection import train_test_split

X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)

print(f'X_train : {X_train.shape}')
print(f'X_test  : {X_test.shape}')


In [ ]:
# ─── Celda 2: Entrenamiento del diccionario sobrecompleto ─────────────────────
from sklearn.decomposition import MiniBatchDictionaryLearning
import time

dico = MiniBatchDictionaryLearning(
    n_components        = 512,
    transform_algorithm = 'lasso_lars',
    transform_alpha     = 1.0,
    alpha               = 1.0,
    batch_size          = 1000,
    max_iter            = 500,
    n_jobs              = -1,
    random_state        = 42,
    verbose             = 0
)

t0 = time.time()
dico.fit(X_train)
print(f'✓ Diccionario entrenado en {time.time()-t0:.1f}s')
print(f'  D : {dico.components_.shape}  →  {dico.components_.shape[0] // dico.components_.shape[1]}× sobrecompleto')


In [ ]:
# ─── Celda 3: Visualización y reproducción de átomos ─────────────────────────
import IPython.display as ipd
from IPython.display import display

# 1. Selección de 10 índices aleatorios
rng     = np.random.default_rng(42)
indices = rng.choice(512, size=10, replace=False)
print(f'Átomos seleccionados: {sorted(indices)}')

def atomo_a_audio(atomo_vec, scaler, n_repeat=50,
                   sr=22050, n_fft=1024, hop_length=256, n_mels=64):
    """
    Convierte un átomo (vector de 64 dims) a señal de audio.
    1. Repite el vector n_repeat veces → espectrograma estático (n_repeat, 64)
    2. Deshace el escalado con inverse_transform
    3. Transpone a (64, n_repeat) para librosa
    4. db_to_power → potencia lineal
    5. mel_to_audio → señal de audio
    """
    # 1. Repetir
    S_rep = np.tile(atomo_vec, (n_repeat, 1))          # (n_repeat, 64)

    # 2. Deshacer escalado
    S_rep = scaler.inverse_transform(S_rep)             # (n_repeat, 64)

    # 3. Transponer
    S_db  = S_rep.T                                     # (64, n_repeat)

    # 4. dB → potencia lineal
    S_pow = librosa.db_to_power(S_db)

    # 5. Mel → audio
    y = librosa.feature.inverse.mel_to_audio(
            S_pow, sr=sr, n_fft=n_fft, hop_length=hop_length)

    # Normalizar para evitar clipping
    y = y / (np.max(np.abs(y)) + 1e-8)
    return y


# ── Cuadrícula 2×5 ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 5, figsize=(18, 6))
fig.suptitle('Perfil Mel de 10 átomos aleatorios del diccionario',
             fontsize=12, fontweight='bold')

audios = []
for ax, idx in zip(axes.flat, indices):
    atomo = dico.components_[idx]                      # vector (64,)
    ax.plot(atomo, color='steelblue', linewidth=1.2)
    ax.set_title(f'Átomo {idx}', fontsize=9, fontweight='bold')
    ax.set_xlabel('Bin Mel'); ax.set_ylabel('Valor')
    ax.grid(alpha=0.3)
    audios.append((idx, atomo_a_audio(atomo, scaler)))

plt.tight_layout()
plt.show()

# ── Reproductores ─────────────────────────────────────────────────────────────
print('\n─── Reproductores de átomos ──────────────────────────────')
for idx, y in audios:
    print(f'Átomo {idx}')
    display(ipd.Audio(y, rate=22050))


In [ ]:
# ─── Celda 4: SparseCoder — Reconstrucción y evaluación ──────────────────────
from sklearn.decomposition import SparseCoder
from sklearn.metrics import mean_squared_error
from IPython.display import display
import IPython.display as ipd

SR_INV     = 22050
N_FFT_INV  = 1024
HOP_INV    = 256

# ── 1. Configurar SparseCoder ─────────────────────────────────────────────────
coder = SparseCoder(
    dictionary          = dico.components_,
    transform_algorithm = 'lasso_lars',
    transform_alpha     = 0.5
)

# ── 2. Seleccionar muestra de test: dígito '1' de 'lucas' ────────────────────
fila = df[(df['digito'] == 1) & (df['persona'] == 'lucas')].iloc[0]
print(f'Muestra seleccionada: {fila.archivo}')

y_orig, _ = librosa.load(fila.ruta, sr=SR_INV, mono=True)

# Espectrograma original con los mismos parámetros del entrenamiento
S_orig = librosa.feature.melspectrogram(
             y=y_orig, sr=SR_INV, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH)
S_orig_db = librosa.power_to_db(S_orig, ref=np.max)   # (64, T)

# Frames normalizados (T, 64)
X_test_frames = scaler.transform(S_orig_db.T)

# ── 3. Códigos dispersos y reconstrucción ────────────────────────────────────
V     = coder.transform(X_test_frames)                 # (T, 512)
X_rec = V @ dico.components_                           # (T, 64)

# Deshacer escalado → dB → potencia
S_rec_db  = scaler.inverse_transform(X_rec).T          # (64, T)
S_rec_pow = librosa.db_to_power(S_rec_db)

# ── 4. Visualización 3 paneles ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
fig.suptitle(f'Sparse Coding — Dígito "{fila.digito}" | {fila.persona}',
             fontsize=12, fontweight='bold')

# Panel 1: Original
img1 = librosa.display.specshow(
           S_orig_db, sr=SR_INV, hop_length=HOP_LENGTH,
           x_axis='time', y_axis='mel', ax=axes[0], cmap='magma')
axes[0].set_title('Original', fontweight='bold')
plt.colorbar(img1, ax=axes[0], format='%+2.0f dB')

# Panel 2: Reconstruido
img2 = librosa.display.specshow(
           S_rec_db, sr=SR_INV, hop_length=HOP_LENGTH,
           x_axis='time', y_axis='mel', ax=axes[1], cmap='magma')
axes[1].set_title('Reconstruido', fontweight='bold')
plt.colorbar(img2, ax=axes[1], format='%+2.0f dB')

# Panel 3: Activaciones (sparsity)
img3 = axes[2].imshow(
           V.T, aspect='auto', origin='lower', cmap='magma',
           interpolation='nearest')
axes[2].set_title('Activaciones V.T\n(líneas finas = alta sparsity)', fontweight='bold')
axes[2].set_xlabel('Frame'); axes[2].set_ylabel('Átomo')
plt.colorbar(img3, ax=axes[2])

plt.tight_layout()
plt.show()

# ── 5. Métricas ───────────────────────────────────────────────────────────────
mse          = mean_squared_error(X_test_frames, X_rec)
atomos_activos = np.mean(np.sum(np.abs(V) > 1e-6, axis=1))
pct_sparsity   = 100 * (1 - atomos_activos / 512)

print(f'MSE de reconstrucción   : {mse:.6f}')
print(f'Átomos activos / frame  : {atomos_activos:.1f} de 512')
print(f'Sparsity                : {pct_sparsity:.1f}%')

# ── Audio original ────────────────────────────────────────────────────────────
print('\nAudio original:')
display(ipd.Audio(y_orig, rate=SR_INV))

# ── Audio reconstruido ────────────────────────────────────────────────────────
y_rec = librosa.feature.inverse.mel_to_audio(
            S_rec_pow, sr=SR_INV, n_fft=N_FFT_INV, hop_length=HOP_INV)
y_rec = y_rec / (np.max(np.abs(y_rec)) + 1e-8)
print('Audio reconstruido:')
display(ipd.Audio(y_rec, rate=SR_INV))
